In [1]:
import pandas as pd
import re
import difflib
from pathlib import Path

# ========= 1. 路径 =========
ANIME_INFO_PATH = "/Users/wangyouqi/Desktop/Yoji_Anime/data/HERE.anime_info.csv"
CHARACTER_PATH = "/Users/wangyouqi/Desktop/Yoji_Anime/data/anime_character_kangkang.csv"

OUTPUT_CLEANED = "/Users/wangyouqi/Desktop/Yoji_Anime/data/anime_character_kangkang_cleaned.csv"
OUTPUT_UNMATCHED = "/Users/wangyouqi/Desktop/Yoji_Anime/data/unmatched_after_cleaning.csv"
OUTPUT_AUDIT = "/Users/wangyouqi/Desktop/Yoji_Anime/data/title_mapping_audit.csv"

# ========= 2. 读文件 =========
anime_df = pd.read_csv(ANIME_INFO_PATH)
char_df = pd.read_csv(CHARACTER_PATH)

# ========= 3. 基础清洗 =========
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def normalize_title(s: str) -> str:
    """
    标题规范化：
    - 转小写
    - 去空格/全角空格
    - 去常见标点
    - 去“第x部/剧场版/特别篇/SP/OVA/OAD”等尾部噪音（尽量温和）
    """
    s = safe_str(s).lower()

    # 全角空格、普通空格
    s = s.replace("\u3000", " ").strip()

    # 统一常见分隔符
    s = s.replace("：", ":")
    s = s.replace("·", "")
    s = s.replace("・", "")
    s = s.replace("　", "")
    s = s.replace(" ", "")

    # 去掉常见括号内容（保守一点，不删主体）
    s = re.sub(r"[\(\[（【].*?[\)\]）】]", "", s)

    # 去掉尾部常见版本词
    s = re.sub(r"(第[0-9一二三四五六七八九十]+部)$", "", s)
    s = re.sub(r"(剧场版|劇場版|特别篇|特別篇|sp|ova|oad|web动画|web動畫)$", "", s)

    # 去掉大多数标点
    s = re.sub(r"[\/\\\-\_\.\,\!\?\~\'\"\|\;]", "", s)
    s = re.sub(r"[:]", "", s)

    return s.strip()

# ========= 4. 手工别名映射（你后面可以持续往里补） =========
MANUAL_MAP = {
    "mutafukaz:operationblackhead": "Mutafukaz",
    "机动战士高达闪光的哈萨维第3部": "机动战士高达 闪光的哈萨维",
    "小熊维尼与跳跳虎": "小熊维尼",
    # 下面这些如果你后续确认 anime_info 里真正对应哪条，再继续补
    # "米奇的圣诞颂歌": "...",
    # "米奇的生日派对": "...",
}

# ========= 5. 构建 anime_info 检索索引 =========
anime_df["name"] = anime_df["name"].apply(safe_str)
anime_df["name_cn"] = anime_df["name_cn"].apply(safe_str)

# 只保留可能有用的列
anime_lookup = anime_df[["id", "name", "name_cn"]].copy()

# 精确 title -> 标准 title
exact_title_map = {}
# 规范化 title -> 标准 title
norm_title_map = {}

for _, row in anime_lookup.iterrows():
    anime_id = row["id"]
    name = row["name"]
    name_cn = row["name_cn"]

    candidates = [name, name_cn]
    for t in candidates:
        if t:
            exact_title_map[t] = {
                "anime_id": anime_id,
                "matched_title": t,
                "anime_name": name,
                "anime_name_cn": name_cn,
            }

            norm_t = normalize_title(t)
            if norm_t and norm_t not in norm_title_map:
                norm_title_map[norm_t] = {
                    "anime_id": anime_id,
                    "matched_title": t,
                    "anime_name": name,
                    "anime_name_cn": name_cn,
                }

# id -> 标准 title
id_map = {}
for _, row in anime_lookup.iterrows():
    id_map[row["id"]] = {
        "anime_id": row["id"],
        "matched_title": row["name_cn"] if row["name_cn"] else row["name"],
        "anime_name": row["name"],
        "anime_name_cn": row["name_cn"],
    }

# 用于 fuzzy 的候选池
all_titles_for_fuzzy = list(set(
    [t for t in anime_lookup["name"].tolist() if t] +
    [t for t in anime_lookup["name_cn"].tolist() if t]
))

normalized_to_originals = {}
for t in all_titles_for_fuzzy:
    nt = normalize_title(t)
    if nt:
        normalized_to_originals.setdefault(nt, []).append(t)

all_normalized_titles = list(normalized_to_originals.keys())

# ========= 6. 匹配函数 =========
def match_anime(row, fuzzy_cutoff=0.92):
    """
    返回：
    - cleaned_anime_title
    - matched_anime_id
    - anime_name
    - anime_name_cn
    - match_method
    """
    subject_id = row["subject_id"]
    raw_title = safe_str(row["anime"])
    norm_raw = normalize_title(raw_title)

    # 0) 手工 alias 先替换
    if norm_raw in MANUAL_MAP:
        alias_target = MANUAL_MAP[norm_raw]
        # alias target 可能是 exact title
        if alias_target in exact_title_map:
            m = exact_title_map[alias_target]
            return pd.Series([
                m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "manual_map"
            ])
        # 也可能是 norm title
        alias_norm = normalize_title(alias_target)
        if alias_norm in norm_title_map:
            m = norm_title_map[alias_norm]
            return pd.Series([
                m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "manual_map"
            ])

    # 1) 用 subject_id 对齐（最稳）
    if pd.notna(subject_id) and subject_id in id_map:
        m = id_map[subject_id]
        return pd.Series([
            m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "subject_id"
        ])

    # 2) 原标题精确匹配
    if raw_title in exact_title_map:
        m = exact_title_map[raw_title]
        return pd.Series([
            m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "exact_title"
        ])

    # 3) 规范化标题匹配
    if norm_raw in norm_title_map:
        m = norm_title_map[norm_raw]
        return pd.Series([
            m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "normalized_title"
        ])

    # 4) 模糊匹配（基于规范化标题）
    # 只在没匹配到时才用，防止误伤
    best = difflib.get_close_matches(norm_raw, all_normalized_titles, n=1, cutoff=fuzzy_cutoff)
    if best:
        best_norm = best[0]
        # 拿这个规范化标题对应的第一个原始标题，再去 exact map 里取标准信息
        candidate_original = normalized_to_originals[best_norm][0]
        if candidate_original in exact_title_map:
            m = exact_title_map[candidate_original]
            return pd.Series([
                m["matched_title"], m["anime_id"], m["anime_name"], m["anime_name_cn"], "fuzzy_title"
            ])

    # 5) 还是不行
    return pd.Series([raw_title, pd.NA, pd.NA, pd.NA, "unmatched"])

# ========= 7. 执行匹配 =========
char_df[[
    "anime_cleaned",
    "matched_anime_id",
    "matched_anime_name",
    "matched_anime_name_cn",
    "match_method"
]] = char_df.apply(match_anime, axis=1)

# ========= 8. 可选：把 anime 字段直接替换成清洗后的标题 =========
# 如果你不想覆盖原字段，可以注释掉下面这行
char_df["anime_original"] = char_df["anime"]
char_df["anime"] = char_df["anime_cleaned"]

# ========= 9. 导出未匹配记录 =========
unmatched_df = char_df[char_df["match_method"] == "unmatched"].copy()

# 聚合看哪些 title 还没处理好
unmatched_summary = (
    unmatched_df.groupby("anime_original", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

# ========= 10. 导出审计表 =========
audit_df = char_df[[
    "subject_id",
    "anime_original",
    "anime",
    "matched_anime_id",
    "matched_anime_name",
    "matched_anime_name_cn",
    "match_method",
    "character",
    "name_cn",
    "cv",
    "relation"
]].copy()

# ========= 11. 保存 =========
char_df.to_csv(OUTPUT_CLEANED, index=False, encoding="utf-8-sig")
unmatched_summary.to_csv(OUTPUT_UNMATCHED, index=False, encoding="utf-8-sig")
audit_df.to_csv(OUTPUT_AUDIT, index=False, encoding="utf-8-sig")

# ========= 12. 打印结果 =========
total_rows = len(char_df)
method_counts = char_df["match_method"].value_counts(dropna=False)

print("清洗完成。")
print(f"总行数: {total_rows}")
print("\n匹配方式统计：")
print(method_counts)

print("\n仍未匹配的动漫名数量：", unmatched_summary.shape[0])
print("输出文件：")
print(f"- {OUTPUT_CLEANED}")
print(f"- {OUTPUT_UNMATCHED}")
print(f"- {OUTPUT_AUDIT}")

清洗完成。
总行数: 79269

匹配方式统计：
match_method
subject_id          79218
unmatched              41
manual_map              9
normalized_title        1
Name: count, dtype: int64

仍未匹配的动漫名数量： 14
输出文件：
- /Users/wangyouqi/Desktop/Yoji_Anime/data/anime_character_kangkang_cleaned.csv
- /Users/wangyouqi/Desktop/Yoji_Anime/data/unmatched_after_cleaning.csv
- /Users/wangyouqi/Desktop/Yoji_Anime/data/title_mapping_audit.csv
